# 📌 Import Necessary Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
from IPython.display import display
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D
import tensorflow as tf
from tensorflow.keras.layers import Input, LSTM, Dense, Dropout
from tensorflow.keras.models import Model, Sequential
import  warnings
warnings.filterwarnings("ignore")
%matplotlib inline

# 📌 Data Preparation and Cleaning

## Load Datasets

In [ ]:
train_df = pd.read_csv("/Users/joyngugi/PycharmProjects/cGAN_LSTM_Clustering_FLFramework/data/rossmann-store-sales/train.csv", parse_dates=['Date'])
test_df = pd.read_csv("/Users/joyngugi/PycharmProjects/cGAN_LSTM_Clustering_FLFramework/data/rossmann-store-sales/test.csv", parse_dates=['Date'])
store_df = pd.read_csv("/Users/joyngugi/PycharmProjects/cGAN_LSTM_Clustering_FLFramework/data/rossmann-store-sales/store.csv")

In [ ]:
print("Train DataFrame Shape:", train_df.shape)
print("Test DataFrame Shape:", test_df.shape)
print("Store DataFrame Shape:", store_df.shape)


In [ ]:
train_df.head(10)

In [ ]:
test_df.head(10)

In [ ]:
store_df.head(10)

## Merge Datasets

In [ ]:
train_merged_df = train_df.merge(store_df, on='Store', how='left')
test_merged_df = test_df.merge(store_df, on='Store', how='left')

In [ ]:
train_merged_df.shape

In [ ]:
train_merged_df.info()

Dataset Description:
- Id - Unique identifier for the row within the test set.
- Store - Unique ID for each store
- DayOfWeek - Day of the week (1 = Monday to 7 = Sunday).
- Date - The specific date of the record.
- Sales - Total sales for that store on the given date.
- Customers - Number of customers on a given day.
- Open - Whether the store was open (1 = Open, 0 = Closed).
- StateHoliday - Indicates a state holiday: '0' = No holiday, 'a' = Public holiday, 'b' = Easter holiday, 'c' = Christmas.
- SchoolHoliday - 	Whether there was a school holiday (1 = Yes, 0 = No).
- StoreType - The type of store (a, b, c, or d).
- Assortment - Assortment level: 'a' = basic, 'b' = extra, 'c' = extended.
- CompetitionDistance - Distance to the nearest competitor store (meters).
- CompetitionOpenSince[Month/Year] - Month/Year the nearest competitor opened.
- Promo - Whether a store is running a promo on that day (1 = Yes, 0 = No).
- Promo2 - Whether the store is participating in the continuing promotion (1 = Yes, 0 = No).
- Promo2Since[Year/Week] - The year/calendar week number when Promo2 started.
- PromoInterval - Describes the consecutive intervals Promo2 is started, naming the months the promotion is started anew. E.g. "Feb,May,Aug,Nov" means each round starts in February, May, August, November of any given year for that store


In [ ]:
train_merged_df.head(50)

## Missing Data

In [ ]:
#Checking for missing values
msno.matrix(train_merged_df)
plt.show()

### ✅ Insights from MSNO Plot
The competition and Promotion columns are the ones with the missing values, lets explore this further to understand how to handle them

In [ ]:
# Quantifying the missingness per column
missing_numbers = train_merged_df.isnull().sum()
missing_percentages = train_merged_df.isnull().mean()
print(f"Number of missing values for each column:\n {missing_numbers}")
print(f"Percentage of missing values for each column:\n {missing_percentages}")

### ✅ Insights from Missing Data Count
- CompetitionOpenSinceMonth and CompetitionOpenSinceMonth **(323348)** have the same number of missing values as well as Promo2SinceWeek,Promo2SinceYear and PromoInterval **(508031)** , CompetitionDistance is **(2642)**.
- Could there be a Correlation between the missing values of Promo2SinceWeek,Promo2SinceYear and PromoInterval and when there was no Promotion i.e. Promo2 = 0? Lets check

### Promotion Missing Data 

In [ ]:
# Create a copy of the dataframe to aid in investigation
train_merged_df_missing = train_merged_df.copy()

In [ ]:
train_merged_df_missing.columns

In [ ]:
# Step 1: Create binary missing indicator columns.
train_merged_df_missing['Promo2SinceWeek_missing'] = train_merged_df_missing['Promo2SinceWeek'].isnull().astype(int)
train_merged_df_missing['Promo2SinceYear_missing'] = train_merged_df_missing['Promo2SinceYear'].isnull().astype(int)
train_merged_df_missing['PromoInterval_missing'] = train_merged_df_missing['PromoInterval'].isnull().astype(int)

# Step 2: Compute correlation matrix with the Promo2 column.
corr_matrix = train_merged_df_missing[['Promo2SinceWeek_missing', 
                        'Promo2SinceYear_missing', 
                        'PromoInterval_missing', 
                        'Promo2']].corr()
print("Correlation Matrix:")
display(corr_matrix)

# Visualise the correlation matrix.
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Between Missing Indicators and Promo2')
plt.show()

# Step 3: Compare missing rates for each Promo2 group.
missing_rates = train_merged_df_missing.groupby('Promo2')[['Promo2SinceWeek_missing', 
                                              'Promo2SinceYear_missing', 
                                              'PromoInterval_missing']].mean()
print("\nMissing Value Rates by Promo2:")
display(missing_rates)


#### ✅ Insights on Promotion Data Correlation Report
- From the above correlation report, the following assumptions can be made:
- The same rows missing for Promo2SinceWeek are the same for Promo2SinceYear as well as PromoInterval.
- The missing values are not missing at Random, there is a perfect alignment of missigness with when there is no promotion.
- Therefore, Promo2SinceWeek and Promo2SinceYear will be imputed with 0 and “No continuing promotion” for PromoInterval

In [ ]:
# Impute in the main dataframe
train_merged_df['Promo2SinceWeek'] = train_merged_df['Promo2SinceWeek'].fillna(0)
train_merged_df['Promo2SinceYear'] = train_merged_df['Promo2SinceYear'].fillna(0)
train_merged_df['PromoInterval']   = train_merged_df['PromoInterval'].fillna("No continuing promotion")


In [ ]:
train_merged_df.isnull().sum()

### Competition Missing Data

In [ ]:
# checking for Patterns 
# Maybe store-specific?
missing_summary = (
    train_merged_df
    .groupby('Store')[['CompetitionDistance', 'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear']]
    .apply(lambda x: x.isnull().sum())
)


stores_with_missing = missing_summary[
    (missing_summary['CompetitionDistance'] > 0) |
    (missing_summary['CompetitionOpenSinceMonth'] > 0) |
    (missing_summary['CompetitionOpenSinceYear'] > 0)
]

pd.set_option('display.max_rows', None)
display(stores_with_missing)



In [ ]:
pd.reset_option('display.max_rows')

In [ ]:
print(store_df[['CompetitionDistance',
                'CompetitionOpenSinceMonth',
                'CompetitionOpenSinceYear']].isnull().sum())


#### ✅ Insights from Competition Missing data
>> From the above the following assumptions can be made:
- assumption1 -> The number of stores with missing data for CompetitionDistance is 3 and, CompetitionOpenSinceMonth and CompetitionOpenSinceYear is 354.
- assumption2 -> The three for CompetitionDistance (291,622,879) also has the same missing values for CompetitionOpenSinceMonth and CompetitionOpenSinceYear **942, 942, and 758** consecutively. What we dont know is if these missing values are on the same records, likely, but we need to confirm.
- assumption -> Both the columns CompetitionOpenSinceMonth and CompetitionOpenSinceYear have the same missing values in all stores, Could they also have the same records with missing values, lets confirm.

#### Testing Assumption2

In [ ]:
# Define the stores you want to check
stores_to_check = [291, 622, 879]

# Filter the DataFrame to only include these stores
subset_1 = train_merged_df[train_merged_df['Store'].isin(stores_to_check)]

# Create missing masks for each column in the subset
mask_distance = subset_1['CompetitionDistance'].isnull()
mask_month = subset_1['CompetitionOpenSinceMonth'].isnull()
mask_year = subset_1['CompetitionOpenSinceYear'].isnull()

# Check if the masks are identical:
aligned = mask_distance.equals(mask_month) and mask_distance.equals(mask_year)
print("Are missing values aligned across all three columns?", aligned)


>> From the above :
- Stores 291,622,879 have mising values for the same records

#### Testing Assumption3

In [ ]:
# Filter the dataframe for rows where missingness doesn't match
mismatch_rows = train_merged_df.loc[
    train_merged_df['CompetitionOpenSinceMonth'].isnull() !=
    train_merged_df['CompetitionOpenSinceYear'].isnull()
]

print("Number of mismatched rows:", len(mismatch_rows))
mismatch_rows.head()


>> From the observations:
>> **All the records with mising values for CompetitionOpenSinceMonth and CompetitionOpenSinceYear are the same across the two columns.**

1. Stores 291, 622, 879 (Group A):

- Missing values across all three competition columns (CompetitionDistance, CompetitionOpenSinceMonth, CompetitionOpenSinceYear) for every row.
- That implies there is zero competition info for these stores.
  
2. For the remaining 351 stores in the missing set (Group B):

- They have CompetitionDistance available, but are missing CompetitionOpenSinceMonth and CompetitionOpenSinceYear.
- So we have a partial picture: we know the distance to a competitor but not the date when that competitor opened.

##### Dealing with the competition missing values

In [ ]:
# For the Group A stores (291, 622, 879):
groupA_stores = [291, 622, 879]
mask_groupA = train_merged_df['Store'].isin(groupA_stores)


# You may keep the missing flags as well:
train_merged_df['HasNoCompetitionData'] = 0
train_merged_df.loc[mask_groupA, 'HasNoCompetitionData'] = 1


In [ ]:
train_merged_df.columns

In [ ]:
train_merged_df.info()

In [ ]:
train_merged_df['CompetitionDistance'] = train_merged_df['CompetitionDistance'].fillna(0)
train_merged_df['CompetitionOpenSinceMonth'] = train_merged_df['CompetitionOpenSinceMonth'].fillna(0)
train_merged_df['CompetitionOpenSinceYear']   = train_merged_df['CompetitionOpenSinceYear'].fillna(0)

In [ ]:
train_merged_df.isnull().sum()

>> For the "Group A" stores that have no competition information recorded, we create a new indicator column "HasNoCompetitionData" and set it to 1. 
We then impute all missing competition-related columns (e.g., CompetitionDistance, CompetitionOpenSinceMonth/Year) with 0 for these stores, ensuring the model can distinguish between genuinely missing data and valid zero values.


##  Duplicates Records

In [ ]:
train_merged_df.duplicated(subset=['Store','Date']).sum()

>> No duplicates for Store–Date combination, the reason this was checked is to make sure there is no sales data for one date recorded more than once for each individual store, as the assumption is that the records are an aggregation of the sales made for different stores for each specific date, so therefore having a store-date combination appering more than could lead to a suspicion of erroneous data entry.

## Data Type Corrections

In [ ]:
train_merged_df.info()

>> Already Parsed date to the correct data type.

# 📌 Exploratory Data Analysis (EDA)

### Distribution Analysis

In [ ]:
StateHoliday_unique_values=train_merged_df["StateHoliday"].unique()
StoreType_unique_values=train_merged_df["StoreType"].unique()
Assortment_unique_values=train_merged_df["Assortment"].unique()

print(f" StateHoliday: {StateHoliday_unique_values} \n StoreType:{StoreType_unique_values} \n Assortment:{Assortment_unique_values}")


In [ ]:
# Change object value with int/float value
train_merged_df["StateHoliday"].replace({'0': 0, 'a': 1, 'b': 2, 'c': 3}, inplace = True)
train_merged_df["StoreType"] = train_merged_df["StoreType"].map({'a': 1, 'b': 2, 'c': 3, 'd': 4})
train_merged_df["Assortment"] = train_merged_df["Assortment"].map({'a': 1, 'b': 2, 'c': 3})
train_merged_df.info()

In [ ]:
train_merged_df['Year'] = train_merged_df['Date'].dt.year
train_merged_df['Month'] = train_merged_df['Date'].dt.month
train_merged_df.head(10)

In [ ]:
train_merged_df.columns

In [ ]:
re_order = ['Store', 'Date','DayOfWeek', 'Month', 'Year', 'Customers', 'Open', 'Promo',
       'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment',
       'CompetitionDistance', 'CompetitionOpenSinceMonth',
       'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek',
       'Promo2SinceYear', 'PromoInterval', 'HasNoCompetitionData', 'Sales']
train_merged_df = train_merged_df[re_order]
train_merged_df.columns

In [ ]:
train_merged_df.info()

In [ ]:
numerical_features = list(train_merged_df.select_dtypes(include=['int64', 'float64', 'int32']).columns)
plt.figure(figsize=(20, 25))

# Loop through each numeric column
for i, col in enumerate(numerical_features, start=1):
    plt.subplot(6, 4, i)  # 6 rows x 4 columns = 20 plots max
    skew_val = train_merged_df[col].skew()  # calculate skewness

    # Plot the distribution
    sns.distplot(train_merged_df[col], kde=True, label=f"Skew = {skew_val:.2f}")
    plt.title(f"Distribution of {col}")
    
    plt.legend(loc="best")
#     plt.xticks(rotation=90)

    # Tight layout helps avoid overlapping text
    plt.tight_layout()

# Display all subplots
plt.show()



#### ✅ Insights from the Distribution Plots

1. Store/ ID Columns

**`Store`**
- **Histogram**: Appears roughly uniform from 1 up to ~1115 (the total number of stores).  
- **Interpretation**: This is effectively an **ID column**, so it’s not really a continuous numeric feature. 


2. Time-Related Columns

 **`DayOfWeek`**
- **Histogram**: Discrete spikes for values 1–7.  
- **Skew**: 0 for a discrete variable.  
- **Interpretation**: The dataset is fairly balanced across the days of the week (though some days may have fewer entries if certain stores close). You might keep this as a **categorical** or do **cyclical encoding** (e.g., sin/cos) for an LSTM or other model.

**`Month`**
- **Histogram**: Spikes at each integer from 1–12.  
- **Skew**: Low at 0.27, but again discrete.  
- **Interpretation**: It’s basically capturing the calendar month of each record. 

**`Year`**
- **Histogram**: Shows 3 discrete spikes (e.g., 2013, 2014, 2015).  
- **Skew**: low at 0.30.  
- **Interpretation**: The dataset covers a few specific years.


3. Sales & Customer Behavior

**`Customers`**
- **Histogram**: Strong **right skew**—a big mass at lower values, with a long tail toward higher customer counts.  
- **Skew**: Quite high, indicating many low/medium values and some extreme outliers.  
- **Interpretation**: Typical for retail data—some days (and some stores) have far more customers than others.  


4. Binary or Categorical Flags

**`Open`, `Promo`, `SchoolHoliday`,  `StateHoliday`,`Promo2`**  
- **Histogram**: Shows two spikes—one at 0, one at 1.  
- **Skew**: Large, because these are mostly 0 or 1.  
- **Interpretation**: 
  - `Open` indicates whether the store was open that day (0=closed, 1=open).  
  - `Promo` indicates if there was an active promotion.  
  - `SchoolHoliday` and `StateHoliday` can influence traffic on weekdays vs. weekends.  
  - `Promo2` is a second type of promotion.  

 **`StoreType`, `Assortment`**
- **Histogram**: Each shows spikes at a few discrete values (e.g., types `a`, `b`, `c`, `d`). If they’re encoded as integers, you’ll see separate peaks.  
- **Skew**: Typically high or undefined because they are **categorical** in nature.  
- **Interpretation**: Tells you what kind of store (like a department, supercenter, etc.) and what assortment (product variety).  
- **Modeling**: Often one-hot encode or use embeddings in a neural network.


5. Competition-Related Columns

**`CompetitionDistance`**
- **Histogram**: Highly **right-skewed** with a large spike near lower distances and a long tail for big distances. 
- **Skew**: 2.93.  
- **Interpretation**: Some stores have a competitor very close by, others are quite far from competition, or the distance might be 0 for those not recorded.  

**`CompetitionOpenSinceMonth`, `CompetitionOpenSinceYear`**
- **Histogram**: Spiky distributions at integer months (1–12) and years (2000+). The spike at 0 is beacause of the missing values which were imputed with 0 representing no/ missing competition.  
- **Skew**: High, given that 0 is overrepresented.  
- **Interpretation**: Many stores have no recorded competitor open date, or the competitor opened prior to the dataset timeframe.  


6. Promo2 Timing

**`Promo2SinceWeek`, `Promo2SinceYear`**
- **Histogram**: Large spike at 0 (stores not in Promo2 or missing data), then discrete spikes for weeks/years.  
- **Skew**: High due to the mass of 0 values.  
- **Interpretation**: Tells you when Promo2 started. If a store never participated, it remains 0.  
- **Modeling**: Similar to competition, you can create “days since promo2 started” or keep them with a missing indicator.


In [ ]:
train_merged_df.info()

#### Encoding Categorical Variables

>>Since there is no natural order to the features **`StoreType`, `Assortment`, `StateHoliday`**, the ordinal approach I used before might be translated wrongly by the model,for example, encoding 'a': 1, 'b': 2, 'c': 3 means the model might treat category 'c' as "larger" or "more important" than 'a' or 'b'. Hence One-Hot encoding which  creates separate binary columns that clearly indicate the presence or absence of each category, avoiding any unintended ordinal implications

In [ ]:
# Reverting back original categories to apply the proper encoding rather than ordinal encoding
train_merged_df["StoreType"] = train_merged_df["StoreType"].map({1:'a', 2:'b', 3:'c', 4:'d'})
train_merged_df["Assortment"] = train_merged_df["Assortment"].map({1:'a', 2:'b', 3:'c'})
train_merged_df["StateHoliday"] = train_merged_df["StateHoliday"].map({0:'0', 1:'a', 2:'b', 3:'c'})

# Create backup of original categorical columns BEFORE encoding
train_merged_df["StoreType_Original"] = train_merged_df["StoreType"]
train_merged_df["Assortment_Original"] = train_merged_df["Assortment"]
train_merged_df["StateHoliday_Original"] = train_merged_df["StateHoliday"]

# Apply One-hot encoding (without dropping original columns explicitly)
train_merged_df = pd.get_dummies(
    train_merged_df,
    columns=["StoreType", "Assortment", "StateHoliday"],
    drop_first=False,
    dtype=int
)


#### Cyclical Encoding

**Why Cyclical Encoding?**

Date-related features (like day-of-week or month) have cyclical characteristics:

- Day-of-week: Sunday (7) is close to Monday (1).
- Months: December (12) is close to January (1).
If they are encoded  as integers (1–7 or 1–12), the model might mistakenly interpret them as linear and non-cyclical (e.g., thinking 7 is far from 1). Cyclical encoding helps models understand the "circular" or periodic nature of time (Kud, 2023).

**How Cyclical Encoding Works**

It uses sine and cosine transformations to preserve cyclical information.
General formula for a cyclical feature:

$$
\text{Feature\_sin} = \sin \left( 2\pi \frac{\text{value}}{\text{max value}} \right)
$$

$$
\text{Feature\_cos} = \cos \left( 2\pi \frac{\text{value}}{\text{max value}} \right)
$$

This transformation maps the features onto a circle, making the first and last values "close" to each other.

In [ ]:
# Month
train_merged_df['Month_sin'] = np.sin(2 * np.pi * train_merged_df['Month'] / 12)
train_merged_df['Month_cos'] = np.cos(2 * np.pi * train_merged_df['Month'] / 12)

# Day of Week
train_merged_df['DayOfWeek_sin'] = np.sin(2 * np.pi * train_merged_df['DayOfWeek'] / 7)
train_merged_df['DayOfWeek_cos'] = np.cos(2 * np.pi * train_merged_df['DayOfWeek'] / 7)


In [ ]:
plt.figure(figsize=(14, 5))

# Original Month distribution
plt.subplot(1, 2, 1)
sns.countplot(x='Month', data=train_merged_df, palette='viridis')
plt.title('Original Month Distribution')

# Month_sin vs Month_cos scatter plot
plt.subplot(1, 2, 2)
sns.scatterplot(
    x='Month_sin', y='Month_cos',
    hue='Month', palette='viridis',
    data=train_merged_df, legend=None
)
plt.title('Cyclical Encoding (Month)')
plt.xlabel('Month_sin')
plt.ylabel('Month_cos')


plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(14, 5))

# Original DayOfWeek distribution
plt.subplot(1, 2, 1)
sns.countplot(x='DayOfWeek', data=train_merged_df, palette='husl')
plt.title('Original DayOfWeek Distribution')

# DayOfWeek cyclical scatter plot
plt.subplot(1, 2, 2)
sns.scatterplot(
    x='DayOfWeek_sin', y='DayOfWeek_cos',
    hue='DayOfWeek', palette='husl',
    data=train_merged_df, legend=None
)
plt.title('Cyclical Encoding (DayOfWeek)')
plt.xlabel('DayOfWeek_sin')
plt.ylabel('DayOfWeek_cos')


plt.tight_layout()
plt.show()


In [ ]:
train_merged_df.columns

In [ ]:
# Extracting DayofYear
train_merged_df['DayOfYear'] = train_merged_df['Date'].dt.dayofyear  
train_merged_df['DayOfYear_sin'] = np.sin(2 * np.pi * train_merged_df['DayOfYear'] / 365)
train_merged_df['DayOfYear_cos'] = np.cos(2 * np.pi * train_merged_df['DayOfYear'] / 365)
    

# 📌 Time Series Analysis

>> "Any series of observations ordered along a single dimension, such as time, may be thought of as a time series. The emphasis in time series analysis is on studying the dependence among observations at different points in time." (Diebold et.al, 2010) 

The purpose of this section will be to explore several aspects both at a global level (the full dataset) and store level (for federated learning purposes):
- Identify Long-term Trends:
Reveal if sales are increasing, decreasing, or stable over time at an aggregate level.

- Detect Seasonality & Cyclical Patterns:
Uncover weekly, monthly, or yearly repeating patterns (e.g., higher sales during holidays or promotions).

- Understand Sales Variability & Stability:
Identify periods of high volatility (e.g., holiday seasons) or stability, guiding the forecasting model decisions.

- Prepare for Model Training:
Provides insights to inform feature engineering, helping decide on appropriate input features (e.g., lagged sales, cyclical features) for the model.


## Overall Sales Trends

In [ ]:
# Aggregate sales data by date
daily_sales = train_merged_df.groupby('Date')['Sales'].sum()

# Plotting overall daily sales
plt.figure(figsize=(16, 6))
daily_sales.plot(color='blue')

# Plot customisation
plt.title('Overall Daily Sales Trends (All Stores)', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Total Daily Sales', fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


### ✅ Insights from the Overall Daily Sales Trend Plot


**Observations:**

- **Clear Weekly Patterns:**  
  - There's a strong, repeating weekly seasonality. Peaks and valleys consistently occur in cycles, suggesting distinct weekday/weekend patterns or promotional cycles.
  - Deep troughs likely represent store closures on Sundays or public holidays.

- **Yearly Seasonal Peaks:**  
  - Significant sales peaks are visible at the end of each year, especially around **December** (holiday seasons like Christmas and New Year).
  - There's a notably large spike at the end of **December 2013**, indicating exceptional sales events, possibly major holiday promotions.

- **Stability Over Time:**  
  - Overall, sales volumes appear stable across years without an evident upward or downward long-term trend, although slight yearly variations are visible.

- **Outliers or Exceptional Events:**  
  - Occasional spikes (e.g., December 2013 and mid-2014) represent unique events or exceptional promotional activities, which may need special attention in modeling.

**Insights :**

- **Weekly seasonality:**  
  Indicates a crucial need to retain or clearly encode day-of-week features .

- **Annual peaks:**  
  Suggest that annual seasonality (using **DayOfYear**) will significantly help the model capture yearly sales spikes.

- **Outliers/Anomalies:**  
  Special spikes or events might require attention either through special flags or careful outlier handling.


## Sales Trends by Store Type

In [ ]:
# Group data by Date and StoreType_Original
store_type_sales = train_merged_df.groupby(['Date', 'StoreType_Original'])['Sales'].sum().reset_index()

# Get unique store types
store_types = store_type_sales['StoreType_Original'].unique()

# Define number of subplots (rows)
n_types = len(store_type_sales['StoreType_Original'].unique())
fig, axes = plt.subplots(nrows=n_types, ncols=1, figsize=(15, 4 * n_types), sharex=False)

# Plot each store type separately
for ax, store_type in zip(axes, store_type_sales['StoreType_Original'].unique()):
    sns.lineplot(
        data=store_type_sales[store_type_sales['StoreType_Original'] == store_type],
        x='Date', y='Sales', ax=ax, color='blue'
    )
    ax.set_title(f'Daily Sales Trends - Store Type {store_type}')
    ax.set_ylabel('Total Sales')
    ax.grid(alpha=0.3)

plt.xlabel('Date')
plt.tight_layout()
plt.show()


### ✅ Insights from Sales Trends by Store Type

Each store type displays unique characteristics in sales patterns, confirming the necessity and value of a federated-clustered modeling approach:

**🔹Store Type A**
- **Clear and strong weekly seasonality** is visible, indicated by the pronounced spikes and consistent dips (likely due to regular weekly closures or lower weekend sales).
- Sales volume is significantly higher compared to other store types, suggesting these are major stores with high sales volume.
- Pronounced spikes around **December** holidays are visible, particularly at the end of 2013 and 2014.

**Insight:**
- Strong weekly and yearly seasonality—essential to capture clearly through cyclical and seasonal features.


**🔹Store Type B**
- Generally lower total sales volume compared to other Stores Types.
- Weekly seasonality appears less pronounced but still clearly present.
- Exhibits a more gradual **upward trend over time**. This indicates growth in average sales from early 2013 toward mid-2015.
- **Moderate volatility** is evident—less pronounced weekly patterns compared to store type A, but still present.

**Insight:**
- Will consider capturing mid-term trends (monthly/quarterly) clearly and ensuring the model can adapt to growth trends and moderate weekly seasonality.


**🔹Store Type C**
- Consistent, clear **weekly seasonality**, similar to Type A.
- A steady level of sales with notable periodic spikes around December, indicative of seasonal or promotional events, though generally smaller peaks compared to Type A.
- Less dramatic annual seasonal spikes compared to A, suggesting either smaller size or fewer promotions/events.

**Insight:**
- Seasonal (weekly) features will be key, but less emphasis might be required on yearly peaks compared to Store Type A.


**🔹Store Type D**
- Clear and stable **weekly pattern** with lower sales volatility compared to Store Types A and B.
- Less prominent end-of-year peaks but clear periodic patterns. Shows relatively stable sales volumes throughout the year, with minor seasonal variations.

**Insight:**
- Capturing weekly seasonal patterns clearly will be essential; however, this type might not require heavy focus on extreme seasonal spikes. Stability is key here.


**Comparative Summary :**

| Store Type | Sales Volume        | Weekly Seasonality? | Yearly Seasonal Peaks? | Volatility |
|------------|---------------------|---------------------|------------------------|------------|
| A          | High                | Very Strong      |  Very Strong (Dec.)  | High       |
| B          | Moderate/Increasing | Moderate         |  Moderate (Dec.)     | Moderate   |
| C          | Moderate            | Strong           |  Moderate            | Moderate   |
| D          | Moderate/Stable     | Clear            |  Less Prominent      | Lower      |


🎯**Key Takeaways for Federated Learning**:

- The **significant variation** in sales patterns across store types strongly supports the **Federated Learning approach**. Each store type clearly represents different distributions of data.
- These insights will greatly inform the choice of **clusters**, the choice of input features, and the tailored LSTM/cGAN models for each cluster.

## Overall Monthly Sales Trend

In [ ]:
# Aggregate sales monthly for each year
monthly_sales = train_merged_df.groupby(['Year', 'Month'])['Sales'].sum().reset_index()

# Plotting sales per year (monthly sales)
plt.figure(figsize=(14, 7))
sns.lineplot(data=monthly_sales, x='Month', y='Sales', hue='Year', marker='o')

# Customie plot
plt.title('Monthly Total Sales Trends per Year', fontsize=16)
plt.xlabel('Month', fontsize=14)
plt.ylabel('Total Monthly Sales', fontsize=14)
plt.xticks(range(1,13))
plt.grid(alpha=0.3)
plt.legend(title='Year')
plt.tight_layout()
plt.show()



### ✅ Insights from Monthly Total Sales Trends

**General Observations**:
- **Seasonality:** 
  - Clear seasonal trends exist—especially noticeable is a significant increase in sales toward the end of each year, peaking notably around **December**.  
  - There's typically a dip mid-year (around June-July), which might correspond with the summer months or vacation seasons.
  
**Yearly Comparisons**:
- **2013**:
  - Starts low, then rises steadily, peaking notably in December (holiday season).
- **2014**:
  - Shows a similar pattern of increase toward the end of the year, with clear seasonal peaks around March-April and especially December.
- **2015**:
  - Limited data (likely due to incomplete year), but shows the typical early-year pattern, starting high (January-February) and potentially continuing upward before data cuts off mid-year.

**Seasonal Patterns Identified**:
- Clear recurring peaks around the **end of the year (December)** each year, highlighting holiday-driven sales.
- Moderate peaks around March-April, likely related to seasonal promotions or events (e.g., Easter).



## Monthly Sales Trend by Store Type

In [ ]:
# Aggregate monthly sales by StoreType_Original and Year
monthly_storetype_sales = train_merged_df.groupby(['Year', 'Month', 'StoreType_Original'])['Sales'].sum().reset_index()

# Get unique store types
store_types = monthly_storetype_sales['StoreType_Original'].unique()

# Plot separate subplots for each Store Type
fig, axes = plt.subplots(len(store_types), 1, figsize=(14, 5 * len(store_types)), sharex=False)

for ax, store_type in zip(axes, store_types):
    sns.lineplot(
        data=monthly_storetype_sales[monthly_storetype_sales['StoreType_Original'] == store_type],
        x='Month', y='Sales', hue='Year', marker='o', ax=ax
    )
    ax.set_title(f'Monthly Sales Trends for Store Type {store_type}', fontsize=16)
    ax.set_xlabel('Month', fontsize=13)
    ax.set_ylabel('Total Monthly Sales', fontsize=13)
    ax.grid(alpha=0.3)
    ax.set_xticks(range(1, 13))
    ax.legend(title='Year')

plt.tight_layout()
plt.show()



### ✅ Insights from Monthly Sales Trends by Store Type

**General Observations Across Store Types:**

- All store types show **clear seasonal patterns**, particularly evident toward the end of the year (November–December) with notable peaks.
- Each store type exhibits distinct patterns, emphasising the value of **federated learning** because it clearly captures different store behaviors.


**Store Type Specific Insights:**

**🔹Store Type A**
- **Sales Volume**: Highest among store types.
- **Seasonality**: Strong **year-end peaks** clearly visible (November–December).
- **Trend**: Sales are stable with no clear upward or downward trend across years.
- **Implication**: Crucial to model strong year-end seasonality clearly. 

**🔹Store Type B**
- **Sales Volume**: Lower but consistent.
- **Seasonality**: Moderate seasonal peaks, especially at year's end (December).
- **Trend**: Clearly increasing sales over years, indicating steady growth.
- **Volatility**: Moderate variability month-to-month.

**🔹Store Type C**
- **Sales Volume**: Moderate, between store types A and B.
- **Seasonality**: Clear annual pattern—higher in March, declines toward mid-year, and peaks strongly again in December.
- **Trend**: Slightly stable, no clear upward or downward annual growth.
- **Volatility**: Moderate, clear annual cycles visible, less stable monthly patterns.

**🔹Store Type D**
- **Sales Volume**: Moderate-to-high.
- **Seasonality**: Strongly defined peak around December each year.
- **Trend**: Generally stable with slight fluctuations, no strong long-term growth or decline visible.


**Comparative Summary:**

| Store Type | Sales Volume | Seasonal Peaks        | Overall Trend            | Monthly Volatility |
|------------|--------------|-----------------------|--------------------------|--------------------|
| A          | High         | Strong (Nov-Dec)      | Stable                   | Moderate to High   |
| B          | Moderate     | Clear, consistent     | Increasing               | Moderate           |
| C          | Moderate     | Clear annual peaks    | Stable                   | Moderate           |
| D          | Moderate     | Strong (December)     | Slight fluctuations      | Lower              |



**🎯Federated Learning Impact:**  
  The clear differences identified strongly justify the use of **clustering** within the federated framework. For example:
  - **Cluster 1**: Store Types with significant growth (Type B) might be modeled separately.
  - **Cluster distinctions** might be based on volume, seasonal behavior, or trend stability.

**🎯Feature Engineering Focus:**
  - Store Type B may require special treatment due to its upward trend.
  - Store Types A and D may benefit from clear seasonality features.
  - Tailored features or separate models for stores with distinct patterns.


## Seasonal Decomposition

Seasonal decomposition breaks down time series data into three essential components:

- Trend: Long-term progression or direction of the sales (e.g., increasing, decreasing, stable).
- Seasonality: Regular, repeating patterns (weekly, monthly, yearly cycles).
- Residuals (Noise): Irregular variations or unexplained fluctuations in the data.

Why Seasonal Decomposition is Important:

- Clearly separates and visualises underlying patterns.
- Helps confirm observations (weekly/yearly seasonality) from the previous plots (Overall Sales Pattern).
- Guides further feature engineering, like identifying important cyclical or seasonal features to use in the forecasting models.


In [ ]:
# Ensure Date is in datetime format and set as index
daily_sales = train_merged_df.groupby('Date')['Sales'].sum()

# Seasonal decomposition (using yearly seasonality: period=365 days)
result = seasonal_decompose(daily_sales, model='additive', period=365)

estimated_trend = result.trend
estimated_seasonal = result.seasonal
estimated_residual = result.resid

# Plot the decomposition components properly
fig, axes = plt.subplots(3, 1, figsize=(18, 10), sharex=True)

axes[0].plot(estimated_trend, label='Trend', color='blue')
axes[0].legend(loc='upper left')
axes[0].set_title('Trend')

axes[1].plot(estimated_seasonal, 'g', label='Seasonality')
axes[1].legend(loc='upper left')
axes[1].set_title('Seasonality')

axes[2].plot(estimated_residual, 'r', label='Residuals')
axes[2].legend(loc='upper left')
axes[2].set_title('Residuals')

# Overall title
plt.suptitle('Seasonal Decomposition of Daily Sales', fontsize=16)

# Show the plot
plt.tight_layout()
plt.show()



### ✅ Insights from Seasonal Decomposition (Overall Daily Sales)**

**Trend Component (Top plot - Blue)**:
- The **trend** initially shows a clear **downward pattern** from early 2013 to early 2014. This indicates that, overall, sales across all stores decreased during that period.
- From early 2014 onward, a clear reversal occurs, with the trend beginning to rise steadily. This indicates recovery and growth in overall sales volume throughout late 2014 and continuing into 2015.
- This indicates clear, long-term sales dynamics: a significant dip followed by a clear, stable recovery.

**Insight for Modeling**:  
- The forecasting model should be capable of capturing shifts in long-term trends. It might benefit from features representing longer-term historical sales performance (lagged or moving averages).


**Seasonality Component (Middle plot - Green)**:
- **Strong, stable seasonal patterns** repeat yearly, clearly visible in consistent spikes and troughs.
- Weekly cycles are very evident—showing regular weekly sales peaks and dips.
- **Annual seasonality** is especially pronounced toward the end of the year (around December), confirming clear yearly recurring promotional or holiday effects.

**Insight for Modeling**:  
- Confirms that weekly and annual cyclical encodings (DayOfWeek, Month, DayOfYear) will be crucial for accurate forecasting.
- Clearly shows that the model must explicitly account for both short-term (weekly) and long-term (yearly) seasonal cycles.


**Residuals Component (Bottom plot - Red)**:
- Residuals represent unexplained or irregular fluctuations after removing seasonal and trend effects.
- A notable anomaly appears in early to mid-2014: a flat segment of residuals, suggesting missing or imputed data (which is not the case) or some issue in that period. It’s critical to investigate this.
- Aside from this anomaly, residuals show consistent variability without obvious patterns, indicating seasonal decomposition has captured most systematic behaviors clearly.

**Insight for Modeling**:  
- Investigate the flat segment in mid-2014 carefully—it may represent data gaps or issues to address before modeling.
- Random fluctuations outside this period confirm no additional clear pattern remains.


**Summary of Key Findings**:

| Component   | Observations                       | Importance for Modeling                  |
|-------------|------------------------------------|------------------------------------------|
| **Trend**       | Initially declining (2013), clearly recovering afterward (2014-2015). | Long-term trend capture, e.g., moving averages |
| **Seasonality** | Strong weekly and yearly seasonal patterns. | Essential cyclical encodings clearly needed|
| **Residuals**   | Generally random fluctuations, notable mid-2014 anomaly. | Investigate anomalies carefully |



## Seasonal decomposition by Store Type

In [ ]:
def seasonal_decompose_storetype(df, store_type, period=365):
    # Filter data by store type
    df_store = df[df['StoreType_Original'] == store_type]

    # Aggregate daily sales
    daily_sales = df_store.groupby('Date')['Sales'].sum()

    # Seasonal decomposition
    result = seasonal_decompose(daily_sales, model='additive', period=period)

    # Extract components
    trend = result.trend
    seasonal = result.seasonal
    residual = result.resid

    # Plot components
    fig, axes = plt.subplots(3, 1, figsize=(18, 10), sharex=True)

    axes[0].plot(trend, label='Trend', color='blue')
    axes[0].legend(loc='upper left')
    axes[0].set_title(f'Trend - Store Type {store_type}')

    axes[1].plot(seasonal, 'g', label='Seasonality')
    axes[1].legend(loc='upper left')
    axes[1].set_title(f'Seasonality - Store Type {store_type}')

    axes[2].plot(residual, 'r', label='Residuals')
    axes[2].legend(loc='upper left')
    axes[2].set_title(f'Residuals - Store Type {store_type}')

    plt.suptitle(f'Seasonal Decomposition of Daily Sales - Store Type {store_type}', fontsize=16)

    plt.tight_layout()
    plt.show()


### Store Type 'a'

In [ ]:
seasonal_decompose_storetype(train_merged_df, 'a')

### Store Type 'b'

In [ ]:
seasonal_decompose_storetype(train_merged_df, 'b')

### Store Type 'c'

In [ ]:
seasonal_decompose_storetype(train_merged_df, 'c')

### Store Type 'd'

In [ ]:
seasonal_decompose_storetype(train_merged_df, 'd')

### ✅ Insights from Seasonal Decomposition by Store Type:**

**🔹Store Type A:**
- **Trend**: 
  - Shows a notable decline from early 2013 until early 2014, followed by a clear recovery starting mid-2014. This indicates a period of downturn followed by sales improvement.
  
- **Seasonality**: 
  - Strong weekly seasonality, evident through consistent and pronounced periodic fluctuations.
  - Clear annual peaks, especially around the holiday seasons.

- **Residuals**:
  - A significant anomaly (flat residuals) in mid-2014 that must be reviewed.
  - Outside this anomaly, residuals show regular variability, suggesting seasonality and trends are generally well-captured.

**Insights for modeling**:
- Critical to explicitly model weekly and annual seasonality.
- Investigate and handle the anomaly in residuals during mid-2014 carefully.


**🔹Store Type B:**
- **Trend**:
  - Clear and consistent upward trend from early 2013 through mid-2015.
  - Indicates strong and sustained growth in sales over the years.

- **Seasonality**:
  - Moderate and stable weekly seasonality.
  - Annual seasonality less pronounced than other types, but still visible.

- **Residuals**:
  - Same mid-2014 anomaly observed (flat residuals). Requires investigation.
  - Residual fluctuations are random, showing that trend and seasonality components capture most of the data clearly.

**Insights for modeling**:
- Emphasise capturing growth trends in the model.
- Moderate seasonal encoding (weekly patterns) will benefit predictions.


**🔹Store Type C:**
- **Trend**:
  - Initial mild downward trend followed by a steady and clear upward trend after early 2014, similar to Type A but less drastic in early decline.

- **Seasonality**:
  - Strong and clear weekly and annual seasonality, similar to store type A.
  - Clear end-of-year peaks.

- **Residuals**:
  - Exhibits a significant anomaly around mid-2014 (flat line).
  - Residuals otherwise indicate random variations clearly captured by trend and seasonal components.

**Insights for modeling**:
- Seasonal patterns must be clearly captured (both weekly and annual).
- Investigate mid-2014 data anomaly carefully.


**🔹Store Type D:**
- **Trend**:
  - Clear downward trend from 2013 to early 2014, followed by evident recovery from mid-2014 onwards (similar to store type A).

- **Seasonality**:
  - Strong weekly and annual seasonality, very consistent.
  - Seasonal spikes clearly around December holidays.

- **Residuals**:
  - Clear anomaly (flat segment) visible again in mid-2014.
  - Apart from this anomaly, residual fluctuations appear random and well-captured by seasonal/trend decomposition.

**Insights for modeling**:
- Essential to include robust seasonal and cyclical features.
- Investigate the residual anomaly around mid-2014 carefully.


**Comparative Summary :**

| Store Type | Trend                                  | Weekly Seasonality | Annual Seasonality | Residuals & Anomalies                 |
|------------|----------------------------------------|--------------------|--------------------|---------------------------------------|
| **A**      | Decreasing then recovering             | Strong             | Strong (Year-end)  | Mid-2014 anomaly, stable residuals    |
| **B**      | Steady increase                        | Moderate           | Moderate           | Mid-2014 anomaly, random residuals    |
| **C**      | Mild dip then clear recovery           | Strong             | Strong (Year-end)  | Mid-2014 anomaly, stable residuals    |
| **D**      | Similar to A, decline followed by rise | Strong             | Strong (Year-end)  | Mid-2014 anomaly, stable residuals    |


## Investigation of the Anomaly in Seasonal Decomposition

In [ ]:
anomaly_period = train_merged_df[
    (train_merged_df['Date'] >= '2014-02-01') & 
    (train_merged_df['Date'] <= '2014-07-01')
]
anomaly_period.head()


In [ ]:
anomaly_period.shape

In [ ]:
zero_sales_days = anomaly_period[anomaly_period['Sales'] == 0]
unscheduled_zero_sales = anomaly_period[(anomaly_period['Sales'] == 0) & (anomaly_period['Open'] == 1)]
store_closed_days = anomaly_period[(anomaly_period['Sales'] == 0) & (anomaly_period['Open'] == 0)]
missing_sales_days = anomaly_period[anomaly_period['Sales'].isnull()]

print(f"Zero sales days: {len(zero_sales_days)}")
print(f"Unscheduled zero sales: {len(unscheduled_zero_sales)}")
print(f"Store closed days: {len(store_closed_days)}")
print(f"Missing sales days: {len(missing_sales_days)}")


In [ ]:
zero_sales_days_global = train_merged_df[train_merged_df['Sales'] == 0]
unscheduled_zero_sales_global = train_merged_df[(train_merged_df['Sales'] == 0) & (train_merged_df['Open'] == 1)]
store_closed_days_global = train_merged_df[(train_merged_df['Sales'] == 0) & (train_merged_df['Open'] == 0)]
missing_sales_days_global = train_merged_df[train_merged_df['Sales'].isnull()]

print(f"Zero sales days: {len(zero_sales_days_global)}")
print(f"Zero sales and closed store days: {len(unscheduled_zero_sales_global)}")
print(f"Store closed days: {len(store_closed_days_global)}")
print(f"Missing sales days: {len(missing_sales_days_global)}")

In [ ]:
zero_sales_by_store = zero_sales_days['Store'].value_counts()
zero_sales_by_store.head(10)


In [ ]:
zero_sales_by_storetype = zero_sales_days['StoreType_Original'].value_counts()
zero_sales_by_storetype


In [ ]:
closed_days = anomaly_period[anomaly_period['Open'] == 0]
closed_days_count = closed_days.groupby('Date').size()
closed_days_count.plot(figsize=(15,5), title="Store Closures (Mid-2014)")
plt.ylabel("Number of Closed Stores")
plt.show()



### ✅ Insights from the Investigation

**1. Store Closures:**
- There is a clear period (around mid-2014) during which a large number of stores (up to **~1000**) were simultaneously closed.  
- This explains the flat residual pattern observed in your earlier seasonal decomposition charts clearly.

**2. Zero Sales Days:**
- Out of **168,185 total records** in the anomaly period, **30,310** (~18%) days had **zero sales**.
- **172,871 total zero sales** in the entire dataset period, indicating stores frequently have zero sales beyond just the anomaly period.
- The anomaly period accounted for about 18% of zero sales days within that specific window, but when considered over the entire dataset, it contributes only a fraction (~17.5%) of all zero-sales instances.
- The dataset contains numerous other days/stores with zero sales, suggesting a regular pattern of store closures (e.g., Sundays, holidays, maintenance days).
- **Store Type Distribution of zero-sales days**:
  - **Store Type A**: **16,563 days**
  - **Store Type D**: **9,633 days**
  - **Store Type C**: **4,114 days**
- Store type **A** clearly experienced the largest impact in terms of zero sales days.

**Summary:**
- Zero-sales events are frequent throughout the dataset, primarily representing routine closures.
- The identified mid-2014 anomaly period remains distinct and noteworthy due to its extensive, prolonged store closures.


## Autocorrelation Analysis (ACF and PACF)

- Autocorrelation, measures how much the past values of a time series affect its future values.
- Imagine tracking daily sales of a store. If sales today are similar to sales yesterday, and yesterday’s sales were similar to the day before, then the data has autocorrelation—meaning past sales influence future sales.
- Autocorrelation Function (ACF) and Partial Autocorrelation Function (PACF) plots help to understand autocorrelation.
- ACF checks the `Overall Influence`: it checks if past values still affect the present, no matter how many steps back they are. It is used to detect seasonality.
- PACF checks the `Direct Influence`: it checks how much each past value affects the present, but only by itself (without interference from other lags). It removes the indirect influence from in-between values.

In [ ]:
# Aggregate daily sales clearly
daily_sales = train_merged_df.groupby('Date')['Sales'].sum()

fig, axes = plt.subplots(2, 1, figsize=(15,10))

# Plot Autocorrelation Function (ACF)
plot_acf(daily_sales, lags=60, ax=axes[0])
axes[0].set_title('Autocorrelation Function (ACF)')

# Plot Partial Autocorrelation Function (PACF)
plot_pacf(daily_sales, lags=60, ax=axes[1])
axes[1].set_title('Partial Autocorrelation Function (PACF)')

plt.tight_layout()
plt.show()



### ✅ Insights from the ACF and PACF Plots

**ACF Plot:**

- **Strong weekly seasonality** is clearly visible:
  - Significant spikes at regular intervals, particularly at **lags of 7, 14, 21, 28, 35, 42, 49, and 56 days**.
  - Indicates strong weekly periodicity in sales—weekly patterns consistently repeat.

- **Gradual Decay**:
  - Autocorrelation decreases gradually but remains significant at weekly intervals.
  - This confirms the importance of incorporating weekly seasonal lags in the model.


**PACF Plot:**

- PACF plot shows **strong significant peaks at initial lags**, particularly at lag 1 and around lag 7:
  - Lag **1** clearly indicates daily dependency (previous day's sales).
  - Lag **7** clearly indicates a strong weekly dependency, confirming weekly cycles.
  
- Beyond lag 7, significance sharply drops off, clearly suggesting that the primary dependencies are on recent and weekly data points.


**What this Means:**

| Insight                  | Action                                         |
|--------------------------|------------------------------------------------|
| Strong Weekly Seasonality| Include weekly cyclical features (lags of 7 days). |
| Daily Dependency         | Include previous-day sales as input.   |
| Sharp Drop after Lag 7   | Focus on short-term (1-7 days) and weekly lags. |

**In conclusion this clearly shows high autocorrelation**



## Autocorrelation Analysis by Store Type

In [ ]:
# Function to perform ACF and PACF analysis by store type
def plot_acf_pacf_by_storetype(df, store_type, lags):
    # Filter data by store type
    store_df = df[df['StoreType_Original'] == store_type]

    # Aggregate daily sales
    daily_sales = store_df.groupby('Date')['Sales'].sum()

    # Plot ACF and PACF
    fig, axes = plt.subplots(2, 1, figsize=(14, 8))

    plot_acf(daily_sales, lags=lags, ax=axes[0])
    axes[0].set_title(f'ACF - Store Type {store_type}')

    plot_pacf(daily_sales, lags=lags, ax=axes[1])
    axes[1].set_title(f'PACF - Store Type {store_type}')

    plt.tight_layout()
    plt.show()


In [ ]:
plot_acf_pacf_by_storetype(train_merged_df, 'a', lags=60)

In [ ]:
plot_acf_pacf_by_storetype(train_merged_df, 'b', lags=60)

In [ ]:
plot_acf_pacf_by_storetype(train_merged_df, 'c', lags=60)

In [ ]:
plot_acf_pacf_by_storetype(train_merged_df, 'd', lags=60)


### ✅ Insights from the ACF and PACF Plots

- **All store types clearly exhibit strong weekly seasonality**.
  - Regular, clear spikes at intervals of **7, 14, 21, 28 days**, reinforcing weekly periodicity.
- Clear daily dependency (lag-1) also strongly visible across store types.


**🔹Store Type a**:
- Strong weekly seasonality (**lags 7, 14, 21...**) clearly evident.
- High correlation clearly seen at lag 1 indicates strong daily dependency.

**🔹Store Type b**:
- Daily autocorrelation (**lags 1, 2, 3…**) clearly strong in the ACF plot, suggesting more significant short-term (daily) dependencies compared to other store types.
- Weekly seasonal peaks remain present but are less dominant.

**🔹Store Type c**:
- Clearly pronounced weekly seasonality.
- PACF confirms significant daily (lag 1) and weekly (lag 7) influences.

**🔹Store Type d**:
- Clearly shows very similar behavior to types a and c, with strong weekly seasonality and daily dependency.


------------------------------------------------------------------------------------------------------------------------

# 📌 Summary of the Time Series Analysis:

1. **Strong Seasonality Assumption:**
   - Weekly patterns are explicitly clear across all store types.
   - Sales at **7-day intervals** strongly influence future sales.

2. **Short-term Dependency Assumption:**
   - Clearly significant correlation with recent sales (lag-1).
   - Immediate past sales clearly impact current demand, validating the use of recent sales as inputs for LSTM models.

3. **Store Type Variability Assumption:**
   - Clear differences in daily dependencies for Store Type **b** versus others, supporting **store-type-specific modeling**.

4. **Data Completeness Assumption:**
   - The mid-2014 anomaly (extended store closures) is explicitly acknowledged but left as-is, assuming minimal disruption to model training and forecasts, as these closures reflect operational realities.

 


| **Assumption**                         | **Derived from Analysis Component**                                     |
|----------------------------------------|-----------------------------------------------------------------------|
| **Strong Seasonality (Weekly)**        | Seasonal Decomposition (clear weekly patterns), ACF & PACF (weekly lags) |
| **Short-term Dependency (lag-1)**      | ACF & PACF Analysis (significant daily lag)                             |
| **Store Type Variability**             | Store-Specific Analysis (different trends, seasonality, autocorrelation patterns) |
| **Data Completeness (Mid-2014 anomaly)**| Overall Sales Trend & Seasonal Decomposition (observed anomaly explicitly acknowledged and documented) |



# 📌 Feature Engineering


Feature engineering is a crucial step in developing an effective predictive model, especially for complex time-series data. It involves creating new features from existing data to:

- Clearly enhance predictive power.
- Explicitly capture patterns discovered in EDA and Time Series Analysis (seasonality, trends, autocorrelation).
- Improve model interpretability and accuracy.


**Feature Engineering Steps:**

1. **Lagged Features (to capture autocorrelation identified)**:
   - Sales from previous days (**Lag 1, 7, 14**).

2. **Rolling Window Features (to capture short-term trends)**:
   - Rolling averages or median of past week/month.

3. **Cyclical Encoding of Time Variables (Already done)**:
   - Day of Week, Day of Month, Month of Year, clearly encoded with sine and cosine transformations.

4. **Store-specific Features (clearly leveraging store characteristics)**:
   - Competition Distance, Promotion Status, Store Type.

5. **Event-based Features**:
   - Clearly encoding holidays or significant store closures.



## Lagged features

Lagged features explicitly leverage the autocorrelation identified earlier in the ACF and PACF analyses. The strongest lags observed were:

- Lag 1: (daily autocorrelation)
- Lag 7: (weekly seasonality)
- Lag 14: (bi-weekly seasonality)


In [ ]:
# Sort by store and date
train_merged_df = train_merged_df.sort_values(['Store', 'Date'])

# Create lagged sales features explicitly
lags = [1, 7, 14]

for lag in lags:
    train_merged_df[f'Sales_Lag_{lag}'] = train_merged_df.groupby('Store')['Sales'].shift(lag)


display(train_merged_df[['Store', 'Date', 'Sales', 'Sales_Lag_1', 'Sales_Lag_7', 'Sales_Lag_14']].head(20))


## Rolling Window Features

Why this is important:
Rolling window features explicitly summarise recent past sales behaviors. They capture trends, momentum, and short-term fluctuations clearly identified in the time-series analysis.

These features typically include:

- Rolling Mean: smooths short-term volatility and captures trends.
- Rolling Median: robust to short-term spikes or outliers.
- Rolling Standard Deviation: measures recent variability in sales.

The rolling windows will include: 
- 7-day window (weekly trend)
- 14-day window (bi-weekly trend)
- 30-day window (monthly trend)

In [ ]:
train_merged_df = train_merged_df.sort_values(['Store', 'Date'])

# Define rolling windows explicitly
rolling_windows = [7, 30]

# Create rolling window features (Mean, Median, Std) for each store explicitly
for window in rolling_windows:
    train_merged_df[f'RollingMean_{window}'] = train_merged_df.groupby('Store')['Sales'].transform(lambda x: x.rolling(window).mean())
    train_merged_df[f'RollingMedian_{window}'] = train_merged_df.groupby('Store')['Sales'].transform(lambda x: x.rolling(window).median())
    train_merged_df[f'RollingStd_{window}'] = train_merged_df.groupby('Store')['Sales'].transform(lambda x: x.rolling(window).std())

display(train_merged_df[['Store', 'Date', 'Sales', 'RollingMean_7', 'RollingMedian_7', 'RollingStd_7']].head(20))


**Handling NaNs from Lag and Rolling Window Features**

When lag features (e.g., lag_1, lag_7, lag_14) are created, the sales series is shifted backward in time to use past values as predictors for the current day’s sales. Naturally, this results in missing values (NaNs) at the start of each store’s time series—because there is no “previous day” data for the first day, no “previous week” data for the first 7 days, and so on.

Similarly, rolling window features (such as 7-day or 14-day rolling means or standard deviations) also introduce NaNs at the beginning of the series. This is because a rolling window needs a full span of prior values to compute an aggregate statistic. For example, a 7-day rolling mean on day 3 cannot be computed accurately, as only three days of data exist—so the result is NaN until all seven prior days are available.

As a result, both lag and rolling window features lead to missing values at the beginning of each store’s time series.


**Options to Handle NaNs**

1. **Imputation**  
   Missing values can be filled using techniques such as forward fill, zero fill, or mean/median imputation. However, this introduces assumptions that may distort real patterns in the data, potentially misleading the model—especially for sensitive architectures like LSTMs, which rely heavily on sequential accuracy.

2. **Deletion**  
   Since there is about about 2.5 years of sales data per store (around 900 days) and deletion will only result into losing up to 30 days by dropping rows that contain NaNs. This is a **small percentage** of the dataset.  The aim is to handle the NaNs in a way to retain a consistent set of valid lag features without introducing artificial patterns via imputation.

Why Deletion Is Preferable Here

- **Small Data Loss**: Dropping 30 days out of ~900 per store is about 1.5% of the data, which typically does not undermine model performance.  
- **Maintains Data Integrity**: Avoid the bias or artifacts that could result from imputation methods.  
- **Simplicity**: It simplifies the pipeline—every row from `day 30` onward has valid lag values and can be used consistently for training and forecasting.


In [ ]:
train_merged_df.isnull().sum()

In [ ]:
train_merged_df.dropna(inplace=True)
train_merged_df.shape

# 📌 Clustering

Clustering involves grouping stores based on their engineered features (lagged, rolling-window, cyclical-encoded) to clearly identify similar temporal behaviors. In this Federated Learning framework, clustering helps to:

- Group stores sharing similar sales behaviors.
- Train more effective, clearly personalised LSTM-cGAN models per cluster.
- Improve accuracy by clearly modeling shared patterns within each cluster.


In [ ]:
train_merged_df.info()

## 1. Store-Level Feature Aggregation

>> Aggregate engineered features to prepare data for clustering.

In [ ]:
store_features = train_merged_df.groupby('Store').agg(
    AvgSalesLag1=('Sales_Lag_1', 'mean'),
    AvgSalesLag7=('Sales_Lag_7', 'mean'),
    AvgSalesLag14=('Sales_Lag_14', 'mean'),
    AvgRollingMean7=('RollingMean_7', 'mean'),
    AvgRollingMean30=('RollingMean_30', 'mean'),
    AvgRollingStd7=('RollingStd_7', 'mean'),
    AvgRollingStd30=('RollingStd_30', 'mean'),
    AvgDayOfWeekSin=('DayOfWeek_sin', 'mean'),
    AvgDayOfWeekCos=('DayOfWeek_cos', 'mean'),
    AvgMonthSin=('Month_sin', 'mean'),
    AvgMonthCos=('Month_cos', 'mean'),
    AvgDayOfYearSin=('DayOfYear_sin', 'mean'),
    AvgDayOfYearCos=('DayOfYear_cos', 'mean')
).reset_index()


In [ ]:
display(store_features)

## 2. Normalisation (Feature Scaling)

>> To standardise features, ensuring each feature has equal importance during clustering.
>> Without normalisation, features with larger scales dominate clustering.

In [ ]:
# Separate Store ID for later reference
store_ids = store_features['Store']

# Select features for scaling 
features_to_scale = store_features.drop('Store', axis=1)

# Apply StandardScaler
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features_to_scale)

# Convert back to DataFrame 
scaled_features_df = pd.DataFrame(scaled_features, columns=features_to_scale.columns)

# Reattach Store IDs 
scaled_features_df['Store'] = store_ids

display(scaled_features_df)


## 3. Determining the Optimal Number of Clusters (Elbow Method)

In [ ]:
# Select scaled features (excluding Store ID)
X = scaled_features_df.drop('Store', axis=1)

# Define the range of clusters to test
wcss = []
k_values = range(1, 10)

# Compute WCSS for each k explicitly
for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X)
    wcss.append(kmeans.inertia_)

# Explicitly plot WCSS vs. number of clusters
plt.figure(figsize=(10, 5))
plt.plot(k_values, wcss, 'bx-', markersize=10)
plt.xlabel('Number of Clusters (k)', fontsize=14)
plt.ylabel('Within-Cluster Sum of Squares (WCSS)', fontsize=14)
plt.title('Elbow Method to Determine Optimal k', fontsize=16)
plt.xticks(k_values)
plt.grid(True)
plt.show()


From this **elbow plot**, it can be seen that the **within‐cluster sum of squares (WCSS)** drops sharply at first and then starts to flatten out. The idea is to find the “elbow,” i.e. the point after which increasing \(k\) yields diminishing returns in terms of reducing WCSS.  

- **K=1 to K=3 or 4**: A steep drop in WCSS, which means each additional cluster is providing substantial improvement.  
- **K=4 or 5 Onward**: The curve starts flattening, so further increases in \(k\) reduce WCSS *less* dramatically.

The elbow seems to be around 3 or 4, so to be sure the Silhoutte score will be computed

### Silhouette Score for (k=3,4,5)

In [ ]:
k_values = [3, 4, 5]

for k in k_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    cluster_labels = kmeans.fit_predict(X)
    
    silhouette_avg = silhouette_score(X, cluster_labels)
    print(f"For n_clusters = {k}, the average silhouette score is: {silhouette_avg:.4f}")


>> This confirms that 3 is the optimal choice for \(k\) as it has the highest silhoutte score.

## 4. Clustering Stores with KMeans (k = 3)

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
store_features['Cluster'] = kmeans.fit_predict(X)

display(store_features[['Store', 'Cluster']].head(10))

In [ ]:
display(store_features['Cluster'].unique())

In [ ]:
# PCA to reduce dimensions to 2 for visualization
pca = PCA(n_components=2, random_state=42)
principal_components = pca.fit_transform(X)

# Create a DataFrame for visualization
pca_df = pd.DataFrame(data=principal_components, columns=['PC1', 'PC2'])
pca_df['Cluster'] = store_features['Cluster'].values

# Plotting
plt.figure(figsize=(10, 7))
sns.scatterplot(
    x='PC1', 
    y='PC2', 
    hue='Cluster', 
    data=pca_df, 
    palette='Set1',
    s=100,
    alpha=0.8
)

plt.title('Store Clusters Visualization (PCA)', fontsize=16)
plt.xlabel('Principal Component 1', fontsize=14)
plt.ylabel('Principal Component 2', fontsize=14)
plt.legend(title='Cluster')
plt.grid(True)
plt.show()


- Clusters are clearly defined with distinct separation.
- Cluster 2 (green) appears particularly well-separated from clusters 0 and 1.
- Cluster 0 (red) and Cluster 1 (blue) show closer proximity and slight overlap, suggesting similarities in characteristics, though still distinct enough to be separate groups.


## 5. Cluster Characterisation

In [ ]:
merged_clustered_df = train_merged_df.merge(store_features[['Store','Cluster']], on='Store', how='left')

In [ ]:
merged_clustered_df.info()

In [ ]:
# Number of Stores per Cluster
cluster_counts = merged_clustered_df.groupby('Cluster')['Store'].nunique()
print(cluster_counts)

In [ ]:
# Create the plot
plt.figure(figsize=(8, 5))
ax = cluster_counts.plot(kind='bar', color='skyblue', edgecolor='black')

# Add labels on each bar
for i, value in enumerate(cluster_counts):
    plt.text(i, value + 0.5, str(value), ha='center', va='bottom', fontsize=10, fontweight='bold')

# Chart formatting
plt.title('Number of Stores per Cluster')
plt.xlabel('Cluster')
plt.ylabel('Number of Stores')
plt.xticks(rotation=0)
plt.grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
# Basic Statistics by Cluster
cluster_summary = merged_clustered_df.groupby('Cluster').agg({
    'Sales': ['mean', 'median', 'min', 'max'],
    'Customers': ['mean', 'median'],
    'CompetitionDistance': ['mean', 'median', 'min', 'max'],
    'Promo': 'mean',
    'Promo2': 'mean'
})
display(cluster_summary)


**Observations:**

- Cluster 1 stores are clearly the highest performing stores on average, characterized by significantly higher average daily sales (8506) and customers (970).

- Cluster 0 stores have lower average daily sales (4934) and fewer customers (543), and these stores are moderately involved in promotions (Promo2 mean: 0.47).

- Cluster 2 stores have moderate sales similar to Cluster 0 but are distinctly characterized by a high level of long-term promotions (Promo2 mean: 0.87), indicating they rely heavily on continuous promotion strategies.

**Insights:**

- Cluster 1 likely represents larger, high-performing stores potentially located in more central or commercially vibrant areas. They rely less on long-term promotions, suggesting that sales are driven more by inherent store popularity, customer loyalty, or prime locations.

- Cluster 0 represents the "average" stores, most common across the dataset, which are moderately successful without extreme reliance on long-term promotions.

- Cluster 2 seems to represent smaller, possibly more competitive or location-challenged stores that rely significantly on sustained promotions (Promo2) to maintain moderate sales performance.

In [ ]:
# Store Type & Assortment Distribution
store_type_distribution = pd.crosstab(merged_clustered_df['Cluster'], merged_clustered_df['StoreType_Original'])
assortment_distribution = pd.crosstab(merged_clustered_df['Cluster'], merged_clustered_df['Assortment_Original'])

display("Store Type Distribution per Cluster:")
display(store_type_distribution)

display("Assortment Distribution per Cluster:")
display(assortment_distribution)


In [ ]:
# Sales Trend by Cluster (Monthly Sales)
monthly_sales_cluster = merged_clustered_df.groupby(['Year', 'Month', 'Cluster'])['Sales'].mean().reset_index()

sns.lineplot(data=monthly_sales_cluster, x='Month', y='Sales', hue='Cluster', style='Year', markers=True)
plt.title('Average Monthly Sales per Cluster')
plt.xlabel('Month')
plt.ylabel('Average Sales')
plt.legend(title='Cluster & Year')
plt.show()


In [ ]:
# Grouping data
monthly_sales_cluster = merged_clustered_df.groupby(['Year', 'Month', 'Cluster'])['Sales'].mean().reset_index()

# Plotting
plt.figure(figsize=(10, 6))
sns.lineplot(data=monthly_sales_cluster, x='Month', y='Sales', hue='Cluster', style='Year', markers=True)

# Title and labels
plt.title('Average Monthly Sales per Cluster')
plt.xlabel('Month')
plt.ylabel('Average Sales')

# Move legend outside the plot
plt.legend(title='Cluster & Year', bbox_to_anchor=(1.05, 1), loc='upper left')

plt.tight_layout()
plt.show()


### ✅ Interpretation of Cluster Characteristics Results:

**1. Number of Stores per Cluster**:
- **Cluster 0 (695 stores)** is the largest, indicating it captures general or average store behaviours.
- **Cluster 1 (241 stores)** and **Cluster 2 (179 stores)** are significantly smaller, suggesting they represent specialised or distinct groups of stores.

**2. Basic Statistics Summary**:
- **Cluster 1** has notably higher average sales (mean = 8506), higher average customer count (970), and lower median competition distance (1880). It likely represents high-performing stores located strategically closer to competitors.
- **Cluster 0** has moderate average sales (mean = 4933) and customer count, and the highest median competition distance (2480), suggesting moderately performing stores potentially located in less competitive areas.
- **Cluster 2** has slightly higher sales (mean = 5406) and a very high Promo2 mean (0.87), indicating stores frequently running continuous promotions.

**3. Store Type and Assortment Distribution**:
- **Cluster 0** predominantly consists of store type 'a' and 'd', signifying these types have general or moderate sales performance.
- **Cluster 1** has a balanced distribution, with store types 'a' and 'd' prominent but notably also includes substantial presence of 'c', suggesting diversified high-performing store types.
- **Cluster 2** has fewer stores overall but primarily consists of type 'a' and 'd', strongly associated with the frequent promo strategy.

- Regarding Assortments:
  - **Cluster 0** heavily features assortment 'a', indicating standard product assortments.
  - **Cluster 1** is notable for a high presence of assortment 'c', this notwithstanding these high-performing stores have more diverse product offerings.
  - **Cluster 2** also has a meaningful share of assortment 'c', aligning with its promotional focus, possibly to support varied products.

**4. Average Monthly Sales per Cluster**:
- **Cluster 1** consistently demonstrates higher monthly sales across all years. This confirms the earlier inference that these stores represent higher-performing segments.
- **Clusters 0 and 2** have lower sales, with cluster 2 slightly outperforming cluster 0, particularly at year-end, likely due to intense promotional activity.
- A clear peak in December sales across all clusters suggests seasonality due to holidays.



**Summary of Cluster Characteristics:**
- **Cluster 0**: Represents general, average-performing stores with standard assortment and fewer competitive pressures.
- **Cluster 1**: High-performing, strategically located, diverse assortment stores.
- **Cluster 2**: Stores heavily reliant on sustained promotions and moderate sales performance.




# 📌 LSTM-CGAN Clustering FL Framework

In [ ]:
merged_clustered_df.columns

In [ ]:
train_ready_df = merged_clustered_df[
  [
    'Store', 'Cluster', 'Date', 
    'DayOfWeek_sin', 'DayOfWeek_cos', 
    'Month_sin', 'Month_cos',
    'DayOfYear_sin', 'DayOfYear_cos',
    'Sales_Lag_1', 'Sales_Lag_7', 'Sales_Lag_14',
    'RollingMean_7', 'RollingMedian_7', 'RollingStd_7',
    'RollingMean_30', 'RollingMedian_30', 'RollingStd_30',
    'CompetitionDistance', 'Promo', 'Promo2',
    'StoreType_a', 'StoreType_b', 'StoreType_c', 'StoreType_d',
    'Assortment_a', 'Assortment_b', 'Assortment_c',
    'Sales'
  ]
]


In [ ]:
train_ready_df

### Partition the Data for FL

**Federated Learning Strategy:**
- Stores (Clients): Each store acts as an individual federated client.

- Clusters: Stores are grouped into clusters based on sales behaviour to reduce data heterogeneity and the Non-IID data problem.

- Local Models: Each store trains a local LSTM model for time series prediction.

- Global Model: Aggregated within each cluster to form cluster-specific global models.

- Conditional GAN (cGAN): Utilised to enhance local data quality or simulate additional data, improving robustness and addressing data scarcity.

In [ ]:
# Partition the data
# Group data by store and cluster
store_clusters = train_ready_df.groupby(['Cluster', 'Store'])

# Structure for storage of local data per store
federated_data = {
    cluster: {store: data for store, data in stores.groupby('Store')}
    for cluster, stores in train_ready_df.groupby('Cluster')
}

# Returns data for store 1 in cluster 0
federated_data[0][1] 


In [ ]:
# Defining Thresholds
MIN_REQUIRED_DATA_POINTS = 365  # Minimum 1 year of data, Able to capture the different trends and seasonality across the year
ZERO_SALES_THRESHOLD = 0.2      # More than 20% zero-sales days
CLUSTER_SIZE_THRESHOLD = 50     # Clusters smaller than 50 stores
VARIANCE_MULTIPLIER = 2.0       # Variance significantly higher than peers
PERFORMANCE_THRESHOLD = 0.75    # Accuracy or metric threshold
ACCEPTABLE_RMSE = 0.15          # Acceptable RMSE after global aggregation


In [ ]:
# Helper Functions to Compute Conditions
def data_quantity_insufficient(store_data):
    return len(store_data) < MIN_REQUIRED_DATA_POINTS

def high_zero_sales_ratio(store_data):
    return (store_data['Sales'] == 0).mean() > ZERO_SALES_THRESHOLD

def is_small_cluster(store_cluster, cluster_store_counts):
    return cluster_store_counts[store_cluster] < CLUSTER_SIZE_THRESHOLD

def high_variance(store_data, cluster_variance_avg):
    return store_data['Sales'].std() > cluster_variance_avg * VARIANCE_MULTIPLIER

def poor_local_performance(store_metrics):
    return store_metrics['accuracy'] < PERFORMANCE_THRESHOLD

def high_post_fed_rmse(store_metrics):
    return store_metrics['rmse'] > ACCEPTABLE_RMSE


In [ ]:
def evaluate_store_for_augmentation(store_id, store_data, store_cluster, cluster_store_counts, cluster_variance_avg, store_metrics):
    """
    Evaluates if a store needs synthetic data via LSTM-cGAN.
    
    Args:
    store_id: ID of the store being evaluated
    store_data: DataFrame containing the data
    store_cluster: Cluster ID for the store
    cluster_store_counts: Dictionary with cluster sizes
    cluster_variance_avg: Average variance for cluster
    store_metrics: Dict with store's local model evaluation metrics
    
    Returns:
    Boolean indicating need for augmentation
    """
    
    if data_quantity_insufficient(store_data):
        print(f"Store {store_id} flagged for insufficient data quantity.")
        return True
    
    if high_zero_sales_ratio(store_data):
        print(f"Store {store_id} flagged for high zero-sales ratio.")
        return True
    
    if is_small_cluster(store_cluster, cluster_store_counts):
        print(f"Store {store_id} flagged for small cluster ({store_cluster}).")
        return True
    
    if high_variance(store_data, cluster_variance_avg):
        print(f"Store {store_id} flagged for high variance in sales.")
        return True
    
    if poor_local_performance(store_metrics):
        print(f"Store {store_id} flagged for poor local performance.")
        return True
    
    if high_post_fed_rmse(store_metrics):
        print(f"Store {store_id} flagged for high post-federation RMSE.")
        return True
    
    # If none of these conditions are met
    print(f"Store {store_id} does not need synthetic augmentation.")
    return False


Defi

# References
Kud, A. (2023) Why We Need Encoding Cyclical Features - Axel Kud, Medium. Available at: https://medium.com/@axelazara6/why-we-need-encoding-cyclical-features-79ecc3531232 (Accessed: 17 March 2025).  
Diebold, F.X., Kilian, L. and Nerlove, M. (2010) ‘Time series analysis’, in Macroeconometrics and Time Series Analysis. London: Palgrave Macmillan UK, pp. 317–342. Available at: https://doi.org/10.1057/9780230280830_35.

